# Knowledge Distillation - a big model teaches a small one

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khanhnd61-vr/modelopt-101/blob/main/examples/02_knowledge_distillation/knowledge_distillation.ipynb)

**Goal.** Train a *tiny* student network that, on its own, would be mediocre - and make
it noticeably better by having a *large* teacher network supervise it. This is
**knowledge distillation (KD)**, one of the basic tools for shrinking models for the edge.

The parts to focus on are:

1. **The teacher-student architecture** - two networks of very different size.
2. **The distillation loss** - how the student learns from the teacher's *soft* predictions.

At the end we **record accuracy and latency before vs. after** so the trade-off is concrete:

| model | size | accuracy | latency |
|-------|------|----------|---------|
| teacher (big) | large | high | slow |
| student, no KD | tiny | lower | fast |
| **student, with KD** | tiny | **higher than no-KD** | fast |

**Runtime:** set `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`.
Full run is a few minutes on a Colab GPU.

## 1. Setup

On Colab `torch`/`torchvision` are pre-installed, so nothing to `pip install`.

In [ ]:
import io, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# --- knobs you can play with ---
EPOCHS     = 15      # epochs for each model (teacher and students)
BATCH_SIZE = 128
LR         = 0.05

## 2. Data - MNIST

10 classes (handwritten digits 0-9) of 28x28 grayscale images. Small and quick to download.

In [ ]:
mean = (0.1307,)
std  = (0.3081,)

train_tf = T.Compose([
    T.RandomCrop(28, padding=2),
    T.ToTensor(),
    T.Normalize(mean, std),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_set = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256,        shuffle=False, num_workers=2)

classes = train_set.classes
print(len(train_set), "train /", len(test_set), "test images")

## 3. Teacher & Student architectures 🔑

The whole idea of KD rests on a **size gap**: a high-capacity **teacher** and a
low-capacity **student** that we actually want to deploy.

- **Teacher**: a VGG-style CNN with three conv stages (64->128->256 channels). Accurate but heavy.
- **Student**: a much smaller CNN (16->32->32 channels). Fast and tiny - but weaker on its own.

Read the channel widths below: that single difference is what makes one the teacher and
the other the student.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )

class Teacher(nn.Module):           # ~1.2M params  (big)
    def __init__(self, num_classes=10):
        super().__init__()
        # TODO: the teacher is the big model - use rising stage widths (e.g. 64 -> 128 -> 256)
        c1, c2, c3 = 16, 16, 16
        self.features = nn.Sequential(
            conv_block(1, c1),   conv_block(c1, c1),   nn.MaxPool2d(2),   # 28 -> 14
            conv_block(c1, c2),  conv_block(c2, c2),   nn.MaxPool2d(2),   # 14 -> 7
            conv_block(c2, c3),  conv_block(c3, c3),   nn.MaxPool2d(2),   # 7  -> 3
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c3, num_classes))

    def forward(self, x):
        return self.head(self.features(x))

class Student(nn.Module):           # ~15K params  (tiny)
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 16),  nn.MaxPool2d(2),   # 28 -> 14
            conv_block(16, 32), nn.MaxPool2d(2),   # 14 -> 7
            conv_block(32, 32), nn.MaxPool2d(2),   # 7  -> 3
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, num_classes))

    def forward(self, x):
        return self.head(self.features(x))

In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

n_teacher = count_params(Teacher())
n_student = count_params(Student())
print(f"teacher params: {n_teacher:,}")
print(f"student params: {n_student:,}")
print(f"compression ratio (params): {n_teacher / n_student:.1f}x smaller")

## 4. Train / eval / metric utilities

Plain training and evaluation loops, a single-image latency probe (batch size 1 on
**CPU**, the realistic edge setting), and an on-disk model-size probe.

In [ ]:
def train(model, loss_fn, epochs=EPOCHS, tag=""):
    """Generic trainer. loss_fn(model_out, x, y) -> scalar loss."""
    model.to(device).train()
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(x), x, y)
            loss.backward()
            opt.step()
        sched.step()
        acc = evaluate(model)
        print(f"[{tag}] epoch {ep+1:02d}/{epochs}  test_acc={acc:.2f}%")
    return model

@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = total = 0
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.numel()
    model.train()
    return 100.0 * correct / total

@torch.no_grad()
def latency_ms(model, n=100):
    """Average single-image inference latency on CPU (ms)."""
    model = copy.deepcopy(model).to("cpu").eval()
    x = torch.randn(1, 1, 28, 28)
    for _ in range(10):            # warmup
        model(x)
    t0 = time.time()
    for _ in range(n):
        model(x)
    return (time.time() - t0) / n * 1000.0

def model_size_mb(model):
    """On-disk size of the saved weights (MB)."""
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6

## 5. Train the teacher

The teacher is just trained normally with cross-entropy. It will be the source of
knowledge for the student.

In [ ]:
teacher = Teacher()
teacher = train(teacher, loss_fn=lambda out, x, y: F.cross_entropy(out, y), tag="teacher")
teacher_acc = evaluate(teacher)
print(f"\nTeacher final accuracy: {teacher_acc:.2f}%")

## 6. Baseline student - *no* distillation

First train the tiny student the ordinary way (plain cross-entropy, no teacher). This is
the **"before"** number we want to beat.

In [ ]:
student_baseline = Student()
student_baseline = train(student_baseline, loss_fn=lambda out, x, y: F.cross_entropy(out, y), tag="student-nokd")
student_baseline_acc = evaluate(student_baseline)
print(f"\nStudent (no KD) accuracy: {student_baseline_acc:.2f}%")

## 7. The distillation loss 🔑

This is the heart of KD. Instead of (or in addition to) the hard one-hot labels, the
student is trained to match the teacher's **soft** probability distribution.

$$
\mathcal{L} \;=\; \underbrace{\alpha\,T^2\;\mathrm{KL}\!\big(\sigma(z_s/T)\,\|\,\sigma(z_t/T)\big)}_{\text{soft targets - learn from the teacher}}
\;+\;
\underbrace{(1-\alpha)\;\mathrm{CE}(z_s,\,y)}_{\text{hard targets - the true labels}}
$$

where $z_s, z_t$ are student/teacher **logits**, $\sigma$ is softmax, $y$ the true label.

- **Temperature $T$** softens the distribution. A high $T$ exposes the teacher's
  *"dark knowledge"* - e.g. that a *cat* image looks a little like a *dog* but nothing
  like a *truck*. Those relative similarities are the extra signal the student gets.
- The $T^2$ factor keeps the gradient magnitude comparable to the cross-entropy term
  (softening by $T$ shrinks gradients by $\sim 1/T^2$).
- **$\alpha$** balances learning from the teacher vs. from the ground-truth labels.

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.7):
    # --- hard targets: ordinary cross-entropy against the true labels ---
    ce_term = F.cross_entropy(student_logits, labels)

    # TODO: soft-target term = KL(log_softmax(student/T) || softmax(teacher/T)) * T**2
    #       use F.kl_div(..., reduction="batchmean")
    kd_term = torch.zeros((), device=student_logits.device)

    # TODO: combine both terms -> alpha * kd_term + (1 - alpha) * ce_term
    return ce_term

## 8. Train the student *with* distillation

Same tiny student architecture, same number of epochs - the **only** change is the loss:
the teacher (frozen, in eval mode) now supervises the student.

In [ ]:
T_KD     = 4.0    # temperature
ALPHA_KD = 0.7    # weight on the teacher (soft) term

teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

def kd_loss_fn(student_out, x, y):
    with torch.no_grad():
        teacher_out = teacher(x)
    return distillation_loss(student_out, teacher_out, y, T=T_KD, alpha=ALPHA_KD)

student_kd = Student()
student_kd = train(student_kd, loss_fn=kd_loss_fn, tag="student-kd")
student_kd_acc = evaluate(student_kd)
print(f"\nStudent (with KD) accuracy: {student_kd_acc:.2f}%")

## 9. Results - size, accuracy & latency, before vs. after

The payoff. The two students are **identical in size and speed**; the only difference is
that one was distilled. Compare its accuracy against the no-KD baseline.

In [ ]:
import matplotlib.pyplot as plt

rows = [
    ("teacher (big)",     n_teacher, model_size_mb(teacher),          teacher_acc,          latency_ms(teacher)),
    ("student, no KD",    n_student, model_size_mb(student_baseline), student_baseline_acc, latency_ms(student_baseline)),
    ("student, with KD",  n_student, model_size_mb(student_kd),       student_kd_acc,       latency_ms(student_kd)),
]

print(f"{'model':<18}{'params':>12}{'size':>10}{'accuracy':>11}{'CPU latency':>14}")
print("-" * 65)
for name, p, sz, acc, lat in rows:
    print(f"{name:<18}{p:>12,}{sz:>8.2f}MB{acc:>10.2f}%{lat:>12.2f}ms")

gain = student_kd_acc - student_baseline_acc
print(f"\n>>> KD gave the tiny student {gain:+.2f}% accuracy at the SAME size & latency.")
print(f">>> The student is {n_teacher/n_student:.1f}x smaller and "
      f"{latency_ms(teacher)/latency_ms(student_kd):.1f}x faster than the teacher.")

# --- plots ---
names  = [r[0] for r in rows]
sizes  = [r[2] for r in rows]
accs   = [r[3] for r in rows]
lats   = [r[4] for r in rows]
colors = ["#888", "#d9534f", "#5cb85c"]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(names, accs,  color=colors); ax[0].set_title("Accuracy (%)"); ax[0].set_ylim(min(accs)-5, 100)
ax[1].bar(names, lats,  color=colors); ax[1].set_title("CPU latency / image (ms)")
ax[2].bar(names, sizes, color=colors); ax[2].set_title("Model size (MB)")
for a in ax: a.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

## 10. Takeaways & things to try

**What you saw:** distillation makes a tiny, fast model more accurate *without changing
its size or speed* - it just learns from a better teacher's soft predictions.

**Experiment (edit and re-run):**
- Change the **temperature** `T_KD` (try `1`, `2`, `8`, `20`). How does the gain change?
- Change **`ALPHA_KD`** (e.g. `0.1` vs `0.9`): more weight on the teacher vs. the labels.
- Make the student **even smaller** (e.g. `8->16` channels). Does KD help more or less?
- Pure soft targets: set `ALPHA_KD = 1.0`. Pure labels: `0.0` (= the no-KD baseline).

**Why it matters for the edge:** KD is often combined with the other techniques here -
you can **distill**, then **quantize** and **prune** the resulting small model.